### Import Dependencies

In [ ]:
import gzip
import json
from itertools import islice
from pathlib import Path

import pandas as pd



In [ ]:
DATA_DIR = Path("../../data")

### Working with downloaded data

In [ ]:
with gzip.open(DATA_DIR/"meta_Electronics.jsonl.gz", "rt") as f:
    first_line = json.loads(f.readline())

In [ ]:
first_line

In [ ]:


with gzip.open(DATA_DIR/"Electronics.jsonl.gz", "rt", encoding="utf-8") as fp:
    for i, line in enumerate(islice(fp, 3), 1):
        print(f"--- Review {i} ---")
        print(json.dumps(json.loads(line), indent=2))

### Filter out only recent items (apeared in stock in 2022 or 2023)

In [ ]:
def filter_data_by_date(data: dict, cutoff_year: int = 2022) -> bool:
    return int(data["details"]["Date First Available"][-4:]) < cutoff_year

In [ ]:
test_dict = {'main_category': 'All Electronics',
 'title': 'FS-1051 FATSHARK TELEPORTER V3 HEADSET',
 'average_rating': 3.5,
 'rating_number': 6,
 'features': [],
 'description': ['Teleporter V3 The “Teleporter V3” kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV headset with integrated NexwaveRF receiver technology while simultaneously recording onboard HD footage with the included “PilotHD” camera. The “Teleporter V3” kit comes complete with everything you need to step into the cockpit of your FPV vehicle. We’ve included our powerful 250mW 5.8Ghz transmitter, 25 degree FOV headset (largest QVGA display available), the brand new “PilotHD” camera with live AV out and all the cables, antennas and connectors needed.'],
 'price': None,
 'images': [{'thumb': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_.jpg',
   'variant': 'MAIN',
   'hi_res': None}],
 'videos': [],
 'store': 'Fat Shark',
 'categories': ['Electronics', 'Television & Video', 'Video Glasses'],
 'details': {'Date First Available': 'August 2, 2014',
  'Manufacturer': 'Fatshark'},
 'parent_asin': 'B00MCW7G9M',
 'bought_together': None}

In [ ]:
filter_data_by_date(test_dict)

In [ ]:
kept = skipped_no_date = 0

with (
    gzip.open(DATA_DIR / "meta_Electronics.jsonl.gz", "rt", encoding="utf-8") as fp,
    open(DATA_DIR / "meta_Electronics_2022_2023.jsonl", "w", encoding="utf-8") as fp_out,
):
    for i, line in enumerate(fp, start=1):
        try:
            is_old = filter_data_by_date(json.loads(line))
        except (KeyError, TypeError, ValueError):
            skipped_no_date += 1
        else:
            if not is_old:
                fp_out.write(line)
                kept += 1

        if i % 100_000 == 0:
            print(f"Processed {i:,} lines")

print(f"Done: kept {kept:,}, skipped (no/invalid date) {skipped_no_date:,}")

### Keep only items that have a main category

In [ ]:
def has_main_category(data: dict) -> bool:
    return data.get("main_category") is not None

In [ ]:
kept = skipped_no_category = 0

with (
    open(DATA_DIR / "meta_Electronics_2022_2023.jsonl", "r", encoding="utf-8") as fp,
    open(DATA_DIR / "meta_Electronics_2022_2023_with_category.jsonl", "w", encoding="utf-8") as fp_out,
):
    for i, line in enumerate(fp, start=1):
        if has_main_category(json.loads(line)):
            fp_out.write(line)
            kept += 1
        else:
            skipped_no_category += 1

        if i % 100_000 == 0:
            print(f"Processed {i:,} lines")

print(f"Done: kept {kept:,}, skipped (no category) {skipped_no_category:,}")

### Explore distribution by categories

In [ ]:
df = pd.read_json(DATA_DIR/"meta_Electronics_2022_2023_with_category.jsonl", lines=True)

In [ ]:
df.head()

In [ ]:
df["main_category"].value_counts().plot(kind="bar")

### Filter out items that have at least 100 ratigs

In [ ]:
df_ratings_100 = df[df["rating_number"] >= 100]

In [ ]:
len(df)

In [ ]:
len(df_ratings_100)

In [ ]:
df_ratings_100["main_category"].value_counts().plot(kind="bar")

### Explore Distribution of ratings

In [ ]:
df_ratings_100["average_rating"].plot(kind="hist", bins=50, range=(0, 5))

### Sample 1000

In [ ]:
df_sample_1000 = df_ratings_100.sample(n=1000, random_state=42)

In [ ]:
len(df_sample_1000)

In [ ]:
df_sample_1000["average_rating"].plot(kind="hist", bins=50, range=(0, 5))

In [ ]:
df_sample_1000["price"].plot(kind="hist", bins=100, range=(0, 500))

In [ ]:
df_sample_1000["main_category"].value_counts().plot(kind="bar")

In [ ]:
df_ratings_100.to_json(DATA_DIR/"meta_Electronics_2022_2023_with_category_ratings_100.jsonl", orient='records', lines=True)
df_sample_1000.to_json(DATA_DIR/"meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", orient='records', lines=True)

### Extract ratings that match sampled data

In [ ]:
df_ratings_100 = pd.read_json(DATA_DIR/"meta_Electronics_2022_2023_with_category_ratings_100.jsonl", lines=True)
df_sample_1000 = pd.read_json(DATA_DIR/"meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

In [ ]:
import duckdb

sampled_asins = df_sample_1000[["parent_asin"]].drop_duplicates()
src = DATA_DIR/"Electronics.jsonl.gz"
dst = DATA_DIR/"Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl"

duckdb.sql(f"""
    COPY (
        SELECT r.*
        FROM read_ndjson_auto('{src}') r
        SEMI JOIN sampled_asins USING (parent_asin)
    ) TO '{dst}' (FORMAT JSON)
""")

In [ ]:
duckdb.sql(f"SELECT * FROM read_ndjson_auto('{DATA_DIR/'Electronics.jsonl.gz'}') LIMIT 5").df()

In [ ]:
duckdb.sql(f"DESCRIBE SELECT * FROM read_ndjson_auto('{src}')").show()